# Lab 03 — Search by Meaning, Not Just Words

A pretrained model may create embeddings. Your job is to implement cosine similarity and deterministic top-*k* retrieval in NumPy.

**Rules:** CPU only; no vector database; no hosted embedding API; no library nearest-neighbour/search implementation.

In [1]:
%pip -q install sentence-transformers numpy

In [2]:
from time import perf_counter
import numpy as np
from sentence_transformers import SentenceTransformer

np.set_printoptions(precision=4, suppress=True)
RNG = np.random.default_rng(42)
MODEL_NAME = "all-MiniLM-L6-v2"

## Challenge corpus

The data is deterministic and deliberately contains paraphrases. It is lab-only data, not the InstaKill domain model.

In [3]:
TOPICS = {
    "food": [
        "A budget-friendly restaurant opened near the station.",
        "Looking for an affordable place to eat dinner tonight.",
        "The cheapest lunch menu I found this week costs under ten euros.",
    ],
    "running": [
        "Easy jogging route beside the river for beginners.",
        "My first 5K training plan starts with short runs and walks.",
        "Anyone know a calm place for an evening run?",
    ],
    "python": [
        "I finally understood why a Python dictionary needs hashable keys.",
        "Small NumPy experiment: vectorized code was much faster than my loop.",
        "Looking for a clear explanation of Python type hints.",
    ],
    "music": [
        "Quiet jazz playlist for a long study session.",
        "Any recommendations for live music this weekend?",
        "This new album is perfect background sound while reading.",
    ],
    "travel": [
        "Best low-cost train trip for a one-day escape?",
        "I need ideas for a weekend hike without a car.",
        "What should I pack for a short mountain trip in autumn?",
    ],
    "books": [
        "Recommend a short science-fiction novel with a strong ending.",
        "I want a book that explains machine learning without too much math.",
        "Which library has a good quiet corner for reading?",
    ],
}

def make_demo_posts(repetitions=200):
    posts = []
    post_id = 1
    for topic, templates in TOPICS.items():
        for repeat in range(repetitions):
            template = templates[repeat % len(templates)]
            posts.append({
                "post_id": post_id,
                "user_id": (repeat % 37) + 1,
                "topic": topic,
                "content": f"{template} [community note {repeat + 1}]",
            })
            post_id += 1
    RNG.shuffle(posts)
    return posts

posts = make_demo_posts()
texts = [post["content"] for post in posts]
assert len(posts) >= 1_000
print(f"{len(posts)} posts; example: {posts[0]}")

1200 posts; example: {'post_id': 70, 'user_id': 33, 'topic': 'food', 'content': 'A budget-friendly restaurant opened near the station. [community note 70]'}


## Baseline: keyword overlap

This baseline is intentionally simple. You may inspect it, but do not use it as your semantic solution.

In [4]:
def keyword_search(query, posts, k=5):
    query_terms = set(query.lower().split())
    scored = []
    for post in posts:
        score = len(query_terms & set(post["content"].lower().split()))
        scored.append((score, post["post_id"], post))
    scored.sort(key=lambda item: (-item[0], item[1]))
    return [post for score, _, post in scored[:k] if score > 0]

keyword_search("cheap dinner", posts, k=3)

[{'post_id': 2,
  'user_id': 2,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 2]'},
 {'post_id': 5,
  'user_id': 5,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 5]'},
 {'post_id': 8,
  'user_id': 8,
  'topic': 'food',
  'content': 'Looking for an affordable place to eat dinner tonight. [community note 8]'}]

## TODO 1 — Encode text

Load the supplied pretrained encoder and create a normalized embedding matrix for all corpus texts. Keep the model call at this boundary; your retrieval functions below must not load a model or perform I/O.

In [5]:
# Model loading and encoding live ONLY in this cell (the I/O / model boundary).
t0 = perf_counter()
model = SentenceTransformer(MODEL_NAME, device="cpu")
load_seconds = perf_counter() - t0

t0 = perf_counter()
corpus_embeddings = model.encode(
    texts,
    batch_size=64,
    normalize_embeddings=True,
    convert_to_numpy=True,
    show_progress_bar=False,
).astype(np.float32)
encode_seconds = perf_counter() - t0

n_params = sum(p.numel() for p in model.parameters())
model_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
index_mb = corpus_embeddings.nbytes / 1e6

print(f"model load: {load_seconds:.1f}s, corpus encode: {encode_seconds:.1f}s for {len(texts)} posts")
print(f"embeddings: shape={corpus_embeddings.shape}, dtype={corpus_embeddings.dtype}, "
      f"row norms in [{np.linalg.norm(corpus_embeddings, axis=1).min():.4f}, "
      f"{np.linalg.norm(corpus_embeddings, axis=1).max():.4f}]")
print(f"model: {n_params / 1e6:.1f}M params, ~{model_mb:.0f} MB fp32; index: {index_mb:.2f} MB")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model load: 6.9s, corpus encode: 18.7s for 1200 posts
embeddings: shape=(1200, 384), dtype=float32, row norms in [1.0000, 1.0000]
model: 22.7M params, ~91 MB fp32; index: 1.84 MB


## TODO 2 — Pure retrieval functions

Implement these functions with NumPy. A zero vector must not produce `NaN` or infinity. For equal scores, the lower `post_id` comes first.

In [6]:
def cosine_similarity(query_vector, corpus_vectors):
    """Return one cosine score for every row in corpus_vectors.

    Pure NumPy, no I/O. Rows (or a query) with zero norm get score 0.0 instead of NaN/inf.
    """
    query = np.asarray(query_vector, dtype=np.float64).ravel()
    corpus = np.atleast_2d(np.asarray(corpus_vectors, dtype=np.float64))
    if corpus.shape[1] != query.shape[0]:
        raise ValueError(f"dimension mismatch: query {query.shape[0]} vs corpus {corpus.shape[1]}")

    denominators = np.linalg.norm(corpus, axis=1) * np.linalg.norm(query)
    dots = corpus @ query
    scores = np.zeros(corpus.shape[0], dtype=np.float64)
    np.divide(dots, denominators, out=scores, where=denominators > 0)
    return np.clip(scores, -1.0, 1.0)  # guard tiny float overshoot


def stable_top_k(scores, post_ids, k):
    """Return indices ordered by score descending, then post_id ascending."""
    scores = np.asarray(scores, dtype=np.float64).ravel()
    post_ids = np.asarray(post_ids).ravel()
    if scores.shape != post_ids.shape:
        raise ValueError("scores and post_ids must have the same length")
    k = max(0, min(int(k), scores.shape[0]))
    # np.lexsort sorts by the LAST key first: primary = -score, secondary = post_id.
    order = np.lexsort((post_ids, -scores))
    return order[:k]


def semantic_search(query_embedding, posts, corpus_embeddings, k=5):
    """Return up to k dictionaries with post data and a numeric score."""
    scores = cosine_similarity(query_embedding, corpus_embeddings)
    post_ids = np.array([post["post_id"] for post in posts])
    return [
        {
            "post_id": posts[i]["post_id"],
            "user_id": posts[i]["user_id"],
            "topic": posts[i]["topic"],
            "content": posts[i]["content"],
            "score": float(scores[i]),
        }
        for i in stable_top_k(scores, post_ids, k)
    ]

## Acceptance checks

Run this cell only after implementing the TODOs. These checks are deliberately small; they verify the retrieval contract, not whether a model is magically perfect.

In [7]:
tiny_query = np.array([1.0, 0.0])
tiny_corpus = np.array([[1.0, 0.0], [0.0, 1.0], [0.5, 0.0], [0.0, 0.0]])
tiny_scores = cosine_similarity(tiny_query, tiny_corpus)
assert np.allclose(tiny_scores, [1.0, 0.0, 1.0, 0.0])
assert np.isfinite(cosine_similarity(np.array([0.0, 0.0]), tiny_corpus)).all()

tie_indices = stable_top_k(np.array([0.7, 0.9, 0.9]), np.array([7, 5, 2]), k=2)
assert list(tie_indices) == [2, 1], "ties must use ascending post_id"

assert corpus_embeddings.shape[0] == len(posts)
assert np.issubdtype(corpus_embeddings.dtype, np.floating)

query_embedding = model.encode(["cheap dinner near the station"], normalize_embeddings=True)[0]
results = semantic_search(query_embedding, posts, corpus_embeddings, k=5)
assert 1 <= len(results) <= 5
assert all({"post_id", "user_id", "content", "score"} <= set(result) for result in results)
assert all(results[i]["score"] >= results[i + 1]["score"] for i in range(len(results) - 1))
assert all(np.isfinite(result["score"]) for result in results)
print("Acceptance checks passed.")

Acceptance checks passed.


## Compare search behavior

Use at least three queries. For each, compare the keyword baseline and your semantic search. Record one useful result and one surprising or weak result.

In [8]:
import re

queries = [
    "cheap dinner",
    "beginner running route",
    "learn Python types",
    "somewhere silent to read",   # extra: paraphrase that spans books / music / running
    "fast numerical code",        # extra: no word overlap with the NumPy post
]
EXPECTED_TOPIC = {
    "cheap dinner": "food",
    "beginner running route": "running",
    "learn Python types": "python",
    "somewhere silent to read": "books",
    "fast numerical code": "python",
}
K = 5


def words(text):
    text = re.sub(r"\[community note \d+\]", "", text.lower())
    return set(re.findall(r"[a-z0-9]+", text))


def precision_at_k(results, topic):
    return sum(r["topic"] == topic for r in results) / K


summary, query_times, zero_overlap_hits = [], [], []
for query in queries:
    topic = EXPECTED_TOPIC[query]
    kw = keyword_search(query, posts, k=K)

    t0 = perf_counter()
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    sem = semantic_search(q_emb, posts, corpus_embeddings, k=K)
    query_times.append(perf_counter() - t0)

    kw_p, sem_p = precision_at_k(kw, topic), precision_at_k(sem, topic)
    summary.append((query, kw_p, sem_p))
    print(f"\n=== {query!r}  (relevant topic: {topic})  keyword P@{K}={kw_p:.2f}  semantic P@{K}={sem_p:.2f}")
    print("  keyword :", [f"{p['post_id']}:{p['content'][:45]}" for p in kw] or "NO RESULTS")
    for r in sem:
        overlap = words(query) & words(r["content"])
        if r["topic"] == topic and not overlap:
            zero_overlap_hits.append((query, r["content"]))
        print(f"  semantic: {r['score']:.4f}  id={r['post_id']:<5} [{r['topic']:<7}] "
              f"{r['content']}  overlap={sorted(overlap) or '-'}")

keyword_mean = np.mean([s[1] for s in summary])
semantic_mean = np.mean([s[2] for s in summary])
distinct_templates = {q: len({r["content"].split(" [")[0] for r in semantic_search(
    model.encode([q], normalize_embeddings=True)[0], posts, corpus_embeddings, k=K)}) for q in queries}

print(f"\nMean topic precision@{K}: keyword={keyword_mean:.2f}, semantic={semantic_mean:.2f}")
print("Distinct underlying templates in semantic top-5:", distinct_templates)
print(f"Relevant semantic hits sharing NO word with the query: {len(zero_overlap_hits)}")
for q, c in zero_overlap_hits[:3]:
    print(f"  {q!r} -> {c}")
assert zero_overlap_hits, "need at least one relevant result that uses different words than the query"


=== 'cheap dinner'  (relevant topic: food)  keyword P@5=1.00  semantic P@5=1.00
  keyword : ['2:Looking for an affordable place to eat dinner', '5:Looking for an affordable place to eat dinner', '8:Looking for an affordable place to eat dinner', '11:Looking for an affordable place to eat dinner', '14:Looking for an affordable place to eat dinner']
  semantic: 0.6392  id=194   [food   ] Looking for an affordable place to eat dinner tonight. [community note 194]  overlap=['dinner']
  semantic: 0.6358  id=197   [food   ] Looking for an affordable place to eat dinner tonight. [community note 197]  overlap=['dinner']
  semantic: 0.6346  id=26    [food   ] Looking for an affordable place to eat dinner tonight. [community note 26]  overlap=['dinner']
  semantic: 0.6319  id=2     [food   ] Looking for an affordable place to eat dinner tonight. [community note 2]  overlap=['dinner']
  semantic: 0.6315  id=137   [food   ] Looking for an affordable place to eat dinner tonight. [community note 13

In [9]:
weakest = min(summary, key=lambda s: s[2])
print(f"""RESULTS
baseline: keyword overlap (set intersection of lowercase tokens), mean topic precision@{K} = {keyword_mean:.2f} over {len(queries)} queries
solution: {MODEL_NAME} normalized embeddings + own NumPy cosine + lexsort top-k (score desc, post_id asc), mean topic precision@{K} = {semantic_mean:.2f}
runtime: model load {load_seconds:.1f}s, encode {len(posts)} posts {encode_seconds:.1f}s on CPU, {1000 * np.mean(query_times):.0f} ms/query (encode + search)
memory / model size: {n_params / 1e6:.1f}M params (~{model_mb:.0f} MB fp32); embedding index {corpus_embeddings.shape} float32 = {index_mb:.2f} MB
3 failure cases:
  1. near-duplicates: top-{K} often repeats one template with different '[community note N]' suffixes (distinct templates: {distinct_templates})
  2. 'cheap dinner' never surfaces the 'budget-friendly restaurant' paraphrase: the literal word 'dinner' wins, so the best paraphrase is buried under {K} copies of one post; the keyword baseline fails differently, matching the stopword 'to' (P@{K} {summary[3][1]:.2f} on {summary[3][0]!r})
  3. ordering among copies of the same sentence is decided by the noise suffix, not by meaning
what I would try next: strip boilerplate like '[community note N]' before encoding; de-duplicate / MMR-diversify results; hybrid score (BM25 + embedding); evaluate with labelled relevance (recall@k, nDCG) on more queries.
""")

RESULTS
baseline: keyword overlap (set intersection of lowercase tokens), mean topic precision@5 = 0.80 over 5 queries
solution: all-MiniLM-L6-v2 normalized embeddings + own NumPy cosine + lexsort top-k (score desc, post_id asc), mean topic precision@5 = 1.00
runtime: model load 6.9s, encode 1200 posts 18.7s on CPU, 49 ms/query (encode + search)
memory / model size: 22.7M params (~91 MB fp32); embedding index (1200, 384) float32 = 1.84 MB
3 failure cases:
  1. near-duplicates: top-5 often repeats one template with different '[community note N]' suffixes (distinct templates: {'cheap dinner': 1, 'beginner running route': 1, 'learn Python types': 1, 'somewhere silent to read': 1, 'fast numerical code': 1})
  2. 'cheap dinner' never surfaces the 'budget-friendly restaurant' paraphrase: the literal word 'dinner' wins, so the best paraphrase is buried under 5 copies of one post; the keyword baseline fails differently, matching the stopword 'to' (P@5 0.00 on 'somewhere silent to read')
  3. o

## RESULTS

The cell above prints this block with the numbers measured on this run (CPU).

```text
RESULTS
baseline: keyword overlap (shared lowercase tokens) -> mean topic precision@5 = 0.80 over 5 queries
solution: all-MiniLM-L6-v2 normalized embeddings + own NumPy cosine similarity + lexsort top-k (score desc, post_id asc) -> mean topic precision@5 = 1.00
runtime: model load 6.9 s, encoding 1200 posts 18.7 s on CPU, 49 ms per query (encode + search)
memory / model size: 22.7M params (~91 MB fp32); embedding index 1200 x 384 float32 = 1.84 MB
3 failure cases:
  1. Near-duplicates: for all 5 queries the semantic top-5 is ONE sentence repeated with different
     "[community note N]" suffixes (1 distinct template per query), so the list has no diversity.
  2. "cheap dinner" returns only "Looking for an affordable place to eat dinner tonight." The closer
     paraphrase "A budget-friendly restaurant opened near the station." never reaches the top-5, because
     the literal word "dinner" dominates. The keyword baseline fails differently: on "somewhere silent
     to read" it matches only the stopword "to" and returns dinner posts (P@5 = 0.00).
  3. The suffix noise decides the order among identical sentences (0.6392 vs 0.6315 for the same text),
     so the ranking within copies is meaningless, though still deterministic.
what I would try next: strip boilerplate like "[community note N]" before encoding; de-duplicate or
MMR-diversify results; hybrid BM25 + embedding scoring; stopword removal for the baseline; evaluate
with labelled relevance (recall@k, nDCG) on more queries.
```

**Useful result:** "somewhere silent to read" -> "Which library has a good quiet corner for reading?" with
zero shared words (silent ~ quiet, somewhere ~ corner/library, read ~ reading). Semantic P@5 is 1.00,
while the keyword baseline scores 0.00.

**Weak result:** "cheap dinner": see failure case 2. All five hits are copies of one post, and the
"budget-friendly restaurant" paraphrase is missing.
